# Bright TV — Exploratory Data Analysis (EDA)

This notebook converts the supplied Bright TV SQL EDA into a Python workflow using **PySpark**. It is designed for use in **Databricks**, where the Bright TV tables are already available.

## Objectives
- Inspect the Bright TV user-profile dataset.
- Measure dataset size and uniqueness.
- Check for duplicate and null `UserID` values.
- Clean and standardize gender, race, and province categories.
- Inspect age range and create age groups.
- Produce simple summary tables suitable for dashboard preparation.

> **Note:** The original SQL queries reference `bright.bright_tv.bright_tv_dataset_user_profiles`. This notebook uses the same table through Spark SQL / PySpark.


In [0]:
# Import libraries
from pyspark.sql import functions as F
from pyspark.sql import types as T


In [0]:
# Load the Bright TV user-profile table
user_profile = spark.read.table("bright.bright_tv.bright_tv_dataset_user_profiles")

display(user_profile)




In [0]:

import pandas as pd
import numpy as np

user_df= user_profile.toPandas()    #convert spart table to pandas dataframe

display(user_df)

## 1. Dataset Size

In [0]:
# Number of rows
#number_of_rows = df.count()
#print(f"Number of rows: {number_of_rows:,}")


user_df.shape #5375 rows, and 9 columns

## 2. Duplicate UserID Check

In [0]:
# 1. Total rows vs unique UserIDs
duplicate_check = pd.DataFrame([{
    "number_of_rows": user_df.shape[0],
    "number_of_unique_userids": user_df["UserID"].nunique()
}])
print(duplicate_check)

# 2. Detailed duplicate UserID check
# Group by UserID and count occurrences
counts = user_df["UserID"].value_counts().reset_index()
counts.columns = ["UserID", "duplicate_cnt"]

# Filter for UserIDs that appear more than once
duplicates = counts[counts["duplicate_cnt"] > 1]

# Print the number of duplicated UserIDs
print(f"Number of duplicated UserIDs: {len(duplicates):,}")
print(duplicates)


## 3. Null UserID Check

In [0]:
# Count rows where UserID is missing
null_userid_count = user_df["UserID"].isna().sum()

print(f"Rows where UserID is NULL: {null_userid_count:,}")


## 4. Gender Analysis and Cleaning

In [0]:
# 1. Get unique gender categories sorted alphabetically
unique_genders = user_df["Gender"].drop_duplicates().sort_values()
print(unique_genders)

# 2. Count NULL values
Gender_null_count = user_df["Gender"].isna().sum()

# 3. Count single-space blank values
Gender_blank_count = (user_df["Gender"] == " ").sum()

# Print results
print(f"NULL gender values: {Gender_null_count:,}")
print(f"Single-space gender values: {Gender_blank_count:,}")


In [0]:
display(user_df)

In [0]:
import numpy as np

# 1. Clean gender values
# Standardize text by converting to lowercase and stripping whitespace
gender_lower_trimmed = user_df["Gender"].astype(str).str.lower().str.strip()

# Apply conditional logic using np.select
conditions = [
    user_df["Gender"].isna(),
    gender_lower_trimmed.isin(["", "none"])
]
choices = ["Unknown", "Unknown"]

user_df["Gender_clean"] = np.select(conditions, choices, default=user_df["Gender"])

# 2. Group by and aggregate summary
Gender_summary = (
    user_df.groupby("Gender_clean")
    .agg(
        subscribers=("UserID", "size"),
        unique_users=("UserID", "nunique")
    )
    .reset_index()
    .sort_values(by="subscribers", ascending=False)
)

print(Gender_summary)


## 5. Race Analysis and Cleaning

In [0]:
# 1. Get unique race categories sorted alphabetically
unique_races = user_df["Race"].drop_duplicates().sort_values()
print(unique_races)

# 2. Count NULL values
race_null_count = user_df["Race"].isna().sum()

# Print result
print(f"NULL race values: {race_null_count:,}")


In [0]:
import numpy as np

# 1. Clean race values
# Standardize text by converting to lowercase and stripping whitespace
race_lower_trimmed = user_df["Race"].astype(str).str.lower().str.strip()

# Apply conditional logic using np.select
conditions = [
    user_df["Race"].isna() | race_lower_trimmed.isin(["", "other", "none"])
]
choices = ["None"]

user_df["race_clean"] = np.select(conditions, choices, default=user_df["Race"])

# 2. Group by and aggregate summary
race_summary = (
    user_df.groupby("race_clean")
    .agg(
        subscribers=("UserID", "size"),
        unique_users=("UserID", "nunique")
    )
    .reset_index()
    .sort_values(by="subscribers", ascending=False)
)

print(race_summary)


## 6. Province Analysis and Cleaning

In [0]:
# Import pandas
import pandas as pd

# Original province categories
print(user_df["Province"].unique())


# Clean province values
user_df["province_clean"] = (
    user_df["Province"]
    .fillna("Uncategorized")
    .str.strip()
    .str.lower()
    .replace("", "Uncategorized")
    .replace("none", "Uncategorized")
)


# Province summary
province_summary = (
    user_df
    .groupby("province_clean")
    .agg(
        subscribers=("UserID", "count"),
        unique_users=("UserID", "nunique")
    )
    .sort_values("subscribers", ascending=False)
    .reset_index()
)

print(province_summary)

## 7. Age Analysis

In [0]:
# Minimum, maximum and average age
age_summary = pd.DataFrame(
    {
        "min_age": [user_df["Age"].min()],
        "max_age": [user_df["Age"].max()],
        "average_age": [user_df["Age"].mean()],
    }
)

print(age_summary)

# Check NULL age values
null_age_count = user_df["Age"].isna().sum()
print(f"NULL age values: {null_age_count:,}")


## 8. Age Grouping

In [0]:
import pandas as pd

# Define the age grouping bins and labels
# pd.cut uses right-inclusive bins by default: (left, right]
# We use -1 as the start so that 0 is caught in the first bin
bins = [-1, 0, 12, 19, 35, 50, 65, float("inf")]
labels = ["Infants", "Kids", "Teens", "Youth", "Adults", "Senior", "Madala"]

# Create age groups based on the supplied logic
# Fill missing ages with a value outside the bins if you want them grouped as 'Madala'
# Or leave them as NaN to be excluded from the summary
user_df["age_group"] = pd.cut(
    user_df["Age"], bins=bins, labels=labels, right=True
)

# Convert to categorical with explicit order for seamless sorting
user_df["age_group"] = pd.Categorical(
    user_df["age_group"], categories=labels, ordered=True
)

# Aggregate and sort automatically by the custom category order
age_group_summary = (
    user_df.groupby("age_group", observed=False)
    .agg(subscribers=("UserID", "nunique"))
    .reset_index()
)

print(age_group_summary)


## 9. Final Clean User-Profile Dataset

In [0]:
# Select useful fields for downstream analysis
user_profiles_clean = user_df[
    [
        "UserID",
        "Gender_summary",
        "race_summary",
        "province_clean",
        "Age",
        "age_group",
    ]
]

print(user_profiles_clean.head(10))


## 10. Data Quality Summary

The following checks reproduce the main validation steps from the original SQL EDA:

- **Row count:** total number of user-profile records.
- **UserID uniqueness:** compares total rows with distinct UserIDs.
- **Duplicate UserIDs:** identifies repeated UserIDs explicitly.
- **Null UserIDs:** confirms whether UserID is missing.
- **Gender:** standardizes blanks and `None` to `Unknown`.
- **Race:** standardizes blank, `None`, and `other` to `None`, following the supplied SQL logic.
- **Province:** standardizes blank and `None` to `Uncategorized`.
- **Age:** checks minimum, maximum, average, and null values.
- **Age groups:** creates Infants, Kids, Teens, Youth, Adults, Senior, and Madala categories.

### Important note
The age-group label **“Madala”** is retained because it appears in the supplied SQL. If this notebook will be shared professionally, consider changing it to a neutral label such as **“66+”** or **“Older Adults”**.
